Imports

In [2]:
import requests
import pandas as pd
import time



help funktion for building the right URL and returns the response as a JSON    

In [3]:
TMDB_API_KEY = "381056fa40bc22a39a9b6c16f58a2465"
BASE_URL = "https://api.themoviedb.org/3"
def tmdb_get(endpoint, params=None, timeout=10):
    if params is None:
        params = {}
    params["api_key"] = TMDB_API_KEY

    url = BASE_URL + endpoint
    r = requests.get(url, params = params, timeout = timeout)
    r.raise_for_status()
    r.status_code
    return r.json()



function that collects movies by iterating through several pages
the movies that are collectet are stored in a list an converted into a Data Frame

In [4]:
def collect_movies(year=2010, pages=3, sort_by="popularity.desc", language="en-US", sleep=0.2):
    all_movies = []

    for page in range(1, pages+1):
        data = tmdb_get(
            "/discover/movie",
            params={
                "primary_release_year": year,
                "sort_by": sort_by,
                "page": page,
                "language": language,
            }
        )
        all_movies.extend(data["results"])
        time.sleep(0.2)
    return pd.DataFrame(all_movies)

collect the movies in a Data Frame

In [5]:
years = range(2000,2025)
frames = []

for y in years:
    frames.append(collect_movies(y,pages=1))

df_movies = pd.concat(frames, ignore_index=True).drop_duplicates("id")

function to get additional movie details

In [6]:
def get_movie_details(movie_id):

    data = tmdb_get(f"/movie/{movie_id}")

    return {
        "id": data.get("id"),
        "budget": data.get("budget"),
        "revenue": data.get("revenue"),
        "runtime": data.get("runtime"),
        "genres": [g["name"] for g in data.get("genres", [])]
    }

goes through all collected movie id´s and request the additional details. 
the results are stored in a list

In [7]:
details = []

for movie_id in df_movies["id"]:

    try:
        d = get_movie_details(movie_id)
        details.append(d)

    except:
        continue

    time.sleep(0.2)

convert collected movie details into a pandas DataFrame

In [8]:
df_details = pd.DataFrame(details)


merge the basic movie data with the detailed movie data unsing the movie id

In [9]:
df_full = df_movies.merge(df_details, on ="id", how="left")
df_full.head()

,adult,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count,budget,revenue,runtime,genres
0,False,/zVsOlg2sNTOJN7RTayC8X2FKjsu.jpg,[53],339308,en,The Unscarred,Four exchange students meet in Berlin 20 years...,24.3563,/tgBRIucjHxtLUXO7HJiK54SteUb.jpg,2000-06-01,The Unscarred,False,5.200,8,0,0,88,[Thriller]
1,False,/uHZRTGMFb1RLmgWcqlIOZsGbDCT.jpg,[35],4247,en,Scary Movie,A familiar-looking group of teenagers find the...,28.9065,/fVQFPRuw3yWXojYDJvA5EoFjUOY.jpg,2000-07-07,Scary Movie,False,6.400,7554,19000000,278023280,88,[Comedy]
2,False,/jhk6D8pim3yaByu1801kMoxXFaX.jpg,"[28, 18, 12]",98,en,Gladiator,"After the death of Emperor Marcus Aurelius, hi...",14.9409,/ty8TGRuvJLPUmAR1H1nRIsgwvim.jpg,2000-05-04,Gladiator,False,8.222,20570,103000000,465516248,155,"[Action, Drama, Adventure]"
3,False,/6OnJoCqeRzoOdzXogm64OD3QGzn.jpg,"[28, 16, 12, 35, 14]",19576,ja,ワンピース,There once was a pirate known as the Great Gol...,11.9091,/aRqQNSuXpcE3dkJC43aEg9f2HXd.jpg,2000-03-04,One Piece: The Movie,False,7.049,381,0,0,51,"[Action, Animation, Adventure, Comedy, Fantasy]"
4,False,/AbFWty0o5nKGo4iLJaGRgqFtC8W.jpg,"[27, 9648]",4234,en,Scream 3,As bodies begin dropping around the Hollywood ...,18.0216,/qpH8ToZVlFD1bakL04LkEKodyDI.jpg,2000-02-04,Scream 3,False,6.004,3947,40000000,161838076,116,"[Horror, Mystery]"


In [10]:
df_full
    


,adult,backdrop_path,genre_ids,id,original_language,original_title,overview,popularity,poster_path,release_date,title,video,vote_average,vote_count,budget,revenue,runtime,genres
0,False,/zVsOlg2sNTOJN7RTayC8X2FKjsu.jpg,[53],339308,en,The Unscarred,Four exchange students meet in Berlin 20 years...,24.3563,/tgBRIucjHxtLUXO7HJiK54SteUb.jpg,2000-06-01,The Unscarred,False,5.200,8,0,0,88,[Thriller]
1,False,/uHZRTGMFb1RLmgWcqlIOZsGbDCT.jpg,[35],4247,en,Scary Movie,A familiar-looking group of teenagers find the...,28.9065,/fVQFPRuw3yWXojYDJvA5EoFjUOY.jpg,2000-07-07,Scary Movie,False,6.400,7554,19000000,278023280,88,[Comedy]
2,False,/jhk6D8pim3yaByu1801kMoxXFaX.jpg,"[28, 18, 12]",98,en,Gladiator,"After the death of Emperor Marcus Aurelius, hi...",14.9409,/ty8TGRuvJLPUmAR1H1nRIsgwvim.jpg,2000-05-04,Gladiator,False,8.222,20570,103000000,465516248,155,"[Action, Drama, Adventure]"
3,False,/6OnJoCqeRzoOdzXogm64OD3QGzn.jpg,"[28, 16, 12, 35, 14]",19576,ja,ワンピース,There once was a pirate known as the Great Gol...,11.9091,/aRqQNSuXpcE3dkJC43aEg9f2HXd.jpg,2000-03-04,One Piece: The Movie,False,7.049,381,0,0,51,"[Action, Animation, Adventure, Comedy, Fantasy]"
4,False,/AbFWty0o5nKGo4iLJaGRgqFtC8W.jpg,"[27, 9648]",4234,en,Scream 3,As bodies begin dropping around the Hollywood ...,18.0216,/qpH8ToZVlFD1bakL04LkEKodyDI.jpg,2000-02-04,Scream 3,False,6.004,3947,40000000,161838076,116,"[Horror, Mystery]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,False,/twsxsfao6ZOVvT8LfudH603MMi6.jpg,"[10751, 35, 16, 878, 28]",519182,en,Despicable Me 4,"Gru and Lucy and their girls—Margo, Edith and ...",18.4099,/wWba3TaojhK7NdycRhoQpsG0FaH.jpg,2024-06-20,Despicable Me 4,False,7.000,3086,100000000,969597394,94,"[Family, Comedy, Animation, Science Fiction, A..."
496,False,/gvLG3Fnznkxl4SmYfcK8gUuqxM8.jpg,"[28, 12, 878]",823464,en,Godzilla x Kong: The New Empire,"Following their explosive showdown, Godzilla a...",17.8354,/z1p34vh7dEOnLDmyCrlUVLuoDzd.jpg,2024-03-27,Godzilla x Kong: The New Empire,False,7.051,4528,150000000,571750016,115,"[Action, Adventure, Science Fiction]"
497,False,/8lqfrOBQ2EcY7znD3LxTYcvlh25.jpg,"[28, 80]",949709,zh,危机航线,"""Anyone? There is ......"" A mysterious message...",16.3272,/v8ySgr4CE45SrrIt4PqqYc1diW6.jpg,2024-09-29,High Forces,False,6.200,36,0,41923655,115,"[Action, Crime]"
498,False,/fK8aA2X9EANs9Bfc5fZ6q9Gj7la.jpg,"[18, 10751]",1278101,zh,小小的我,"Liu Chunhe, suffering from cerebral palsy, bra...",17.5721,/rLj287mA1tp4hZmd69umb1rOZu8.jpg,2024-12-27,Big World,False,8.000,56,7000000,109500000,131,"[Drama, Family]"


In [11]:
df_full.to_excel("movies_dataset.xlsx", index=False)

ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
381056fa40bc22a39a9b6c16f58a2465